<a href="https://colab.research.google.com/github/AnnyNny/sixray-kd/blob/anna-local-work/anna_student_one_box.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

True
Tesla T4


In [12]:
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
print("WANDB_API_KEY is set:", bool(os.environ.get("WANDB_API_KEY")))

WANDB_API_KEY is set: True


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [17]:
%cd /content

!rm -rf /content/sixray-kd
!git clone https://github.com/AnnyNny/sixray-kd.git /content/sixray-kd

%cd /content/sixray-kd
!git checkout anna-local-work

!git status
!git log --oneline -5

/content
Cloning into '/content/sixray-kd'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 133 (delta 40), reused 115 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 7.81 MiB | 14.28 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/sixray-kd
Branch 'anna-local-work' set up to track remote branch 'anna-local-work' from 'origin'.
Switched to a new branch 'anna-local-work'
On branch anna-local-work
Your branch is up to date with 'origin/anna-local-work'.

nothing to commit, working tree clean
167012c (HEAD -> anna-local-work, origin/anna-local-work) Fix wandb login helper
dd6c594 Fix wandb login for Colab script execution
e4bba30 Add one-box ablation for ResNet18 student
82ce0f7 Add README.md
22c5362 Add smoke test for ResNet18 student modules


In [4]:

!bash /content/sixray-kd/scripts/setup_colab.sh

/content/sixray-kd
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
Dataset 'subset_clean.zip' extracted to /content/data


In [6]:
!pip install -q torchmetrics pycocotools wandb timm evaluate transformers

In [7]:
from pathlib import Path

paths = [
    Path("/content/data/train/images"),
    Path("/content/data/train.json"),
    Path("/content/data/test/images"),
    Path("/content/data/test.json"),
    Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json"),
]

for path in paths:
    print(path, "OK" if path.exists() else "MISSING")

/content/data/train/images OK
/content/data/train.json OK
/content/data/test/images OK
/content/data/test.json OK
/content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json OK


In [8]:
%env SIXRAY_ABLATION=baseline
!python students/anna_student_resnet18/scripts/smoke_test_student.py

env: SIXRAY_ABLATION=baseline
Device: cuda
IMAGE_SIZE: 640
GRID_SIZE: 20
NUM_BOXES: 2
NUM_CLASSES: 5

Building model...
Total parameters: 12952660
Trainable parameters: 12952660

Running dummy forward pass...
Predictions shape: (2, 20, 20, 20)
Expected shape: (2, 20, 20, 20)

Encoding dummy targets...
Objectness target: (2, 2, 1, 20, 20)
Bbox target: (2, 2, 4, 20, 20)
Class target: (2, 2, 20, 20)
Positive mask: (2, 2, 1, 20, 20)
Assigned objects: 2
Skipped objects: 0

Computing dummy loss...
Loss: 2.8110971450805664
Loss dict: {'objectness_loss': 0.7374467253684998, 'bbox_loss': 0.049307312816381454, 'class_loss': 1.5805773735046387, 'total_loss': 2.8110971450805664}

Decoding dummy predictions...
Decoded images: 2
Image 0: boxes=(94, 4), scores=(94,), labels=(94,)
Image 1: boxes=(89, 4), scores=(89,), labels=(89,)

Smoke test passed.


In [18]:
%env SIXRAY_ABLATION=one_box
!python students/anna_student_resnet18/scripts/train_student.py

env: SIXRAY_ABLATION=one_box
Device: cuda
Use AMP: True
Ablation: one_box
Ablation description: Ablation with one box slot per grid cell instead of two.

Building datasets...
Loaded split: /content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json
Train: 10500
Val: 1500
Test: 13246
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 10500
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 1500
Dataset: /content/data/test.json
Images folder: /content/data/test/images
Images used here: 13246

Building dataloaders...

Building model...
Total parameters: 12950090
Trainable parameters: 12950090

Initializing W&B...
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anna_smetanina (anna_smet

In [19]:
%env SIXRAY_ABLATION=one_box
!python students/anna_student_resnet18/scripts/evaluate_student.py

env: SIXRAY_ABLATION=one_box
Device: cuda
Use AMP: True

Building datasets...
Loaded split: /content/drive/MyDrive/DatasetAPAI/SIXray_Project/splits/split_seed42_train10500_val300pos1200neg.json
Train: 10500
Val: 1500
Test: 13246
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 10500
Dataset: /content/data/train.json
Images folder: /content/data/train/images
Images used here: 1500
Dataset: /content/data/test.json
Images folder: /content/data/test/images
Images used here: 13246

Building dataloaders...

Building model...
Total parameters: 12950090
Trainable parameters: 12950090

Loading checkpoint:
/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/one_box/student_resnet18_yolo2_local_offsets_one_box_best_map_50.pth
Loaded epoch: 41

Evaluating val

val loss:
{'total_loss': 0.505502735747187, 'objectness_loss': 0.2602425646762896, 'bbox_loss': 0.003075215156226477, 'class_loss': 0.21450801952827522, 'assigned

In [ ]:
import json
from pathlib import Path

path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/one_box/student_resnet18_eval_metrics.json")

with open(path, "r") as f:
    results = json.load(f)

print("VAL")
print("mAP:", results["val"]["detection"]["map"])
print("mAP@50:", results["val"]["detection"]["map_50"])
print("mAP@75:", results["val"]["detection"]["map_75"])
print("per-class:", results["val"]["detection"]["map_per_class"])

print("\nTEST")
print("mAP:", results["test"]["detection"]["map"])
print("mAP@50:", results["test"]["detection"]["map_50"])
print("mAP@75:", results["test"]["detection"]["map_75"])
print("per-class:", results["test"]["detection"]["map_per_class"])

In [4]:
import json
import pandas as pd
from pathlib import Path

class_names = ["gun", "knife", "wrench", "pliers", "scissors"]

baseline_path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/student_resnet18_eval_metrics.json")
one_box_path = Path("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/ablations/one_box/student_resnet18_eval_metrics.json")

with open(baseline_path, "r") as f:
    baseline = json.load(f)

with open(one_box_path, "r") as f:
    one_box = json.load(f)


def get_detection_metrics(result, split="test"):
    split_data = result[split]
    if "detection" in split_data:
        return split_data["detection"]
    if all(k in split_data for k in ["map", "map_50", "map_75"]):
        return split_data


rows = []

for name, k, result in [
    ("Baseline K=2", 2, baseline),
    ("One-box K=1", 1, one_box),
]:
    test_det = get_detection_metrics(result, split="test")

    row = {
        "model": name,
        "boxes_per_cell": k,
        "mAP": test_det["map"],
        "mAP@50": test_det["map_50"],
        "mAP@75": test_det["map_75"],
    }

    for class_name, value in zip(class_names, test_det["map_per_class"]):
        row[f"AP_{class_name}"] = value

    rows.append(row)

df = pd.DataFrame(rows)
df

,model,boxes_per_cell,mAP,mAP@50,mAP@75,AP_gun,AP_knife,AP_wrench,AP_pliers,AP_scissors
0,Baseline K=2,2,0.292397,0.553677,0.274398,0.600464,0.217004,0.175816,0.231006,0.237693
1,One-box K=1,1,0.310381,0.549394,0.309042,0.623167,0.265045,0.172125,0.262255,0.229316


In [5]:
df.to_csv("/content/drive/MyDrive/DatasetAPAI/SIXray_Project/student_checkpoints_anna/one_box_ablation_comparison.csv", index=False)